---
numbering: false
---

# 2.4: Lines and planes in ℝ³

:::{warning} Under construction
This section is still under construction.
:::


In [1]:
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
import sys

_notes_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'myst.yml').exists())
if str(_notes_root) not in sys.path:
    sys.path.insert(0, str(_notes_root))
from plot_style import style_plotly

BLUE, ORANGE, PINK = '#3d81f6', 'orange', '#d81a60'


def base3():
    fig=style_plotly(go.Figure(),renderer='plotly_mimetype')
    axis=dict(range=[-6,8],dtick=2,showbackground=True,showspikes=False,
              backgroundcolor='white',gridcolor='#e5e7eb',zerolinecolor='#9ca3af')
    fig.update_layout(autosize=True,height=520,showlegend=False,font=dict(size=16),
                      margin=dict(l=0,r=0,t=15,b=0),
                      scene=dict(bgcolor='white',xaxis=dict(title='x',**axis),yaxis=dict(title='y',**axis),
                                 zaxis=dict(title='z',**axis),aspectmode='cube',
                                 camera=dict(eye=dict(x=1.6,y=-2.1,z=1.3))))
    for direction in np.eye(3):
        line3(fig,-6*direction,8*direction,'#9ca3af',width=2)
    return fig


def line3(fig,start,end,color=BLUE,width=5,dash='solid'):
    fig.add_trace(go.Scatter3d(x=[start[0],end[0]],y=[start[1],end[1]],z=[start[2],end[2]],
                              mode='lines',line=dict(color=color,width=width,dash=dash),
                              hoverinfo='skip',showlegend=False))


def vec3(fig,end,label,color=BLUE,start=(0,0,0),offset=(0.25,0.25,0.35)):
    start,end=np.asarray(start,float),np.asarray(end,float)
    direction=(end-start)/np.linalg.norm(end-start)
    line3(fig,start,end-0.15*direction,color,width=7)
    fig.add_trace(go.Cone(x=[end[0]],y=[end[1]],z=[end[2]],u=[direction[0]],v=[direction[1]],w=[direction[2]],
                         anchor='tip',sizemode='absolute',sizeref=0.45,colorscale=[[0,color],[1,color]],
                         showscale=False,hoverinfo='skip'))
    pos=end+offset
    fig.add_trace(go.Scatter3d(x=[pos[0]],y=[pos[1]],z=[pos[2]],mode='text',text=[label],
                              textfont=dict(family='Palatino',color=color,size=20),hoverinfo='skip'))


def plane3(fig,normal,color=BLUE,opacity=0.24,d=0):
    # Solve n_x*x + n_y*y + n_z*z = d for y on a rectangular x,z mesh.
    x,z=np.meshgrid(np.linspace(-5,7,2),np.linspace(-5,7,2))
    n=np.asarray(normal,float)
    y=(d-n[0]*x-n[2]*z)/n[1]
    fig.add_trace(go.Surface(x=x.tolist(),y=y.tolist(),z=z.tolist(),
                            colorscale=[[0,color],[1,color]],opacity=opacity,
                            showscale=False,hoverinfo='skip'))
    corners=np.array([[x[0,0],y[0,0],z[0,0]],[x[0,1],y[0,1],z[0,1]],
                      [x[1,1],y[1,1],z[1,1]],[x[1,0],y[1,0],z[1,0]]])
    for i in range(4): line3(fig,corners[i],corners[(i+1)%4],color,width=2)


In [Chapter 2.1](02-01.ipynb), we learned how to describe a line in $\mathbb R^2$ using either a vector along it or a vector perpendicular to it. Both descriptions gave us the same line, but each made different questions easier to answer.

We'll use the same ideas here, with one extra coordinate. Taking scalar multiples of a nonzero vector still gives us a line. Taking linear combinations of two independent vectors gives us a **plane**. Our goal is to connect that picture to an equation involving $x$, $y$, and $z$.

**Drag the 3D figures to view the lines and planes from different angles.**


---

## Lines through the origin

Remember that the span of one vector is the set of all its scalar multiples. Let's start with

$$\vec v=\begin{bmatrix}3\\4\\5\end{bmatrix}.$$

Multiplying $\vec v$ by a scalar changes its length and possibly reverses its direction. If we draw all of these multiples from the origin, their tips trace out a line! As before, we can write this line as

$$\ell=\operatorname{span}(\vec v)
=\left\{t\begin{bmatrix}3\\4\\5\end{bmatrix}:t\in\mathbb R\right\}.$$

We can also write this in **parametric form**:

$$\begin{bmatrix}x\\y\\z\end{bmatrix}
=t\begin{bmatrix}3\\4\\5\end{bmatrix},\qquad t\in\mathbb R.$$

For instance, $t=1$ gives $(3,4,5)$, $t=-1$ gives $(-3,-4,-5)$, and $t=0$ gives the origin. The parameter $t$ can be any real number; the figure only shows part of the line.


In [2]:
fig=base3()
v=np.array([3,4,5])
line3(fig,-1.1*v,1.3*v,BLUE,width=4)
vec3(fig,v,'<i>v</i>⃗',BLUE)
fig.show()


```{figure} #plot-24-line
:label: fig-24-line
:class: course-caption
:alt: A line through the origin with direction vector v.

The line through the origin spanned by
$\vec v=\begin{bmatrix}3\\4\\5\end{bmatrix}$.
```


---

## Planes as spans

To fill a plane through the origin, we need two nonzero vectors that are not scalar multiples of each other. We'll call these two vectors **independent**: neither direction can be obtained by scaling the other.

For example, let

$$\vec v_1=\begin{bmatrix}3\\4\\5\end{bmatrix},\qquad
\vec v_2=\begin{bmatrix}2\\0\\-2\end{bmatrix}.$$

These vectors are independent: every multiple of $\vec v_2$ has a second entry of $0$, while $\vec v_1$ has a second entry of $4$. So, there's no number we can multiply $\vec v_2$ by to get $\vec v_1$.

Notice that we're asking for less than we did in [Chapter 2.2](02-02.ipynb). Our vectors don't need to have length one, and they don't need to be perpendicular. We just need two independent directions in the plane.


In [3]:
fig=base3()
plane3(fig,(1,-2,1))
vec3(fig,(3,4,5),'<i>v</i>⃗<sub>1</sub>',BLUE)
vec3(fig,(2,0,-2),'<i>v</i>⃗<sub>2</sub>',ORANGE,offset=(-0.8,0.2,0.4))
fig.show()


```{figure} #plot-24-span
:label: fig-24-span
:class: course-caption
:alt: Two independent vectors span the blue plane through the origin.

The vectors $\vec v_1$ and $\vec v_2$ determine a plane $P$ through
the origin.
```


Think of $\vec v_1$ and $\vec v_2$ as two directions we're allowed to move in. We can travel any amount along $\ell_1=\operatorname{span}(\vec v_1)$, then any amount along $\ell_2=\operatorname{span}(\vec v_2)$. By varying these amounts, we can reach any point in the plane.

To see this in the picture, pick a vector $\vec w$ in the plane and complete a parallelogram with sides along $\ell_1$ and $\ell_2$.

The sides of the parallelogram give us $\vec w_1=a\vec v_1$ and $\vec w_2=b\vec v_2$ for some scalars $a,b$. Adding them gives

$$\vec w=\vec w_1+\vec w_2=a\vec v_1+b\vec v_2.$$

For example, choosing $a=1$ and $b=-1$ gives

$$\vec w=\vec v_1-\vec v_2
=\begin{bmatrix}3\\4\\5\end{bmatrix}-\begin{bmatrix}2\\0\\-2\end{bmatrix}
=\begin{bmatrix}1\\4\\7\end{bmatrix}.$$

In this example, $\vec w_1=\vec v_1$ and $\vec w_2=-\vec v_2$. They add up to $\vec w$, but they aren't perpendicular. So, don't confuse them with $\vec w_{\parallel}$ and $\vec w_{\perp}$ from [Chapter 2.3](02-03.ipynb): here, we're completing a parallelogram, which need not be a rectangle.


In [4]:
fig=base3()
plane3(fig,(1,-2,1),opacity=0.15)
v1=np.array([3,4,5]); v2=np.array([2,0,-2]); w=v1-v2
line3(fig,-1.1*v1,1.3*v1,BLUE,width=3)
line3(fig,-2.5*v2,3.5*v2,ORANGE,width=3)
line3(fig,v1,w,'gray',width=4,dash='dash'); line3(fig,-v2,w,'gray',width=4,dash='dash')
vec3(fig,v1,'<i>w</i>⃗<sub>1</sub>',BLUE,offset=(0.4,0.4,0.5))
vec3(fig,-v2,'<i>w</i>⃗<sub>2</sub>',ORANGE,offset=(-0.8,-0.6,-0.5))
vec3(fig,w,'<i>w</i>⃗',PINK,offset=(-0.5,-0.6,0.5))
fig.show()


```{figure} #plot-24-parallelogram
:label: fig-24-parallelogram
:class: course-caption
:alt: Blue and orange components complete a parallelogram and add to the pink vector w.

The exact parallelogram relation
$\vec w=\vec w_1+\vec w_2=\vec v_1-\vec v_2$.
```


:::{note} Definition: Span of two vectors
The **span** of $\vec v_1$ and $\vec v_2$ is the set of all their linear combinations:

$$\operatorname{span}(\vec v_1,\vec v_2)
=\{a\vec v_1+b\vec v_2:a,b\in\mathbb R\}.$$

If the two vectors are independent, their span is a plane through the origin.
:::

In our example, this gives

$$\begin{aligned}
P&=\operatorname{span}(\vec v_1,\vec v_2)\\
&=\left\{a\begin{bmatrix}3\\4\\5\end{bmatrix}
+b\begin{bmatrix}2\\0\\-2\end{bmatrix}:a,b\in\mathbb R\right\}\\
&=\left\{\begin{bmatrix}3a+2b\\4a\\5a-2b\end{bmatrix}:a,b\in\mathbb R\right\}.
\end{aligned}$$

This is a **parametric description** of $P$. You can think of $a$ and $b$ as two knobs we can turn: $a$ tells us how much of $\vec v_1$ to use, and $b$ tells us how much of $\vec v_2$ to use. Letting both range over all real numbers gives the entire plane.

::::{tip} Activity 1
Use this last description to write out some other vectors in the plane.

:::{tip} Solution
:class: dropdown

Two examples are

$$\vec v_3=\begin{bmatrix}1\\4\\7\end{bmatrix},\qquad
\vec v_4=\begin{bmatrix}12\\8\\4\end{bmatrix}.$$

There are infinitely many correct answers! We just need to pick values of $a$ and $b$ and substitute them into our expression. For instance, $a=1$ and $b=-1$ gives

$$\vec v_3=\vec v_1-\vec v_2
=\begin{bmatrix}3(1)+2(-1)\\4(1)\\5(1)-2(-1)\end{bmatrix}
=\begin{bmatrix}1\\4\\7\end{bmatrix}.$$

Taking $a=2$ and $b=3$ gives

$$\vec v_4=2\vec v_1+3\vec v_2
=\begin{bmatrix}3(2)+2(3)\\4(2)\\5(2)-2(3)\end{bmatrix}
=\begin{bmatrix}12\\8\\4\end{bmatrix}.$$

We don't need a separate check that these vectors are in $P$: we made them by taking linear combinations of $\vec v_1$ and $\vec v_2$, which is exactly what it means to be in their span.
:::
::::


:::{note} Different vectors spanning the same plane
:class: dropdown

The vectors from our solution to Activity 1,

$$\vec v_3=\begin{bmatrix}1\\4\\7\end{bmatrix},\qquad
\vec v_4=\begin{bmatrix}12\\8\\4\end{bmatrix},$$

also span $P$:

$$P=\operatorname{span}(\vec v_1,\vec v_2)=\operatorname{span}(\vec v_3,\vec v_4).$$

The key idea is that **each pair can be built from the other pair**. That means anything we can build using one pair can also be built using the other. Let's check this carefully.

We already know that

$$\vec v_3=\vec v_1-\vec v_2,\qquad
\vec v_4=2\vec v_1+3\vec v_2,$$

so any linear combination of $\vec v_3$ and $\vec v_4$ is also a linear combination of $\vec v_1$ and $\vec v_2$:

$$\begin{aligned}
s\vec v_3+t\vec v_4
&=s(\vec v_1-\vec v_2)+t(2\vec v_1+3\vec v_2)\\
&=(s+2t)\vec v_1+(-s+3t)\vec v_2.
\end{aligned}$$

So every vector in $\operatorname{span}(\vec v_3,\vec v_4)$ belongs to $P$.

But that's only half of what we need. We also need to show that we can build $\vec v_1$ and $\vec v_2$ using $\vec v_3$ and $\vec v_4$. Notice that

$$\begin{aligned}
3\vec v_3+\vec v_4
&=3(\vec v_1-\vec v_2)+(2\vec v_1+3\vec v_2)=5\vec v_1,\\
\vec v_4-2\vec v_3
&=(2\vec v_1+3\vec v_2)-2(\vec v_1-\vec v_2)=5\vec v_2
\end{aligned}$$

so

$$\vec v_1=\frac35\vec v_3+\frac15\vec v_4,\qquad
\vec v_2=-\frac25\vec v_3+\frac15\vec v_4.$$

Therefore, any vector in $P$ can also be written as

$$\begin{aligned}
a\vec v_1+b\vec v_2
&=a\left(\frac35\vec v_3+\frac15\vec v_4\right)
+b\left(-\frac25\vec v_3+\frac15\vec v_4\right)\\
&=\frac{3a-2b}{5}\vec v_3+\frac{a+b}{5}\vec v_4.
\end{aligned}$$

So, we can move back and forth between the two descriptions. The coefficients change, but the set of vectors we can reach stays the same!

More generally, any two independent vectors **in $P$** span $P$. As in Chapter 2.1, a span description is not unique.

:::


In [5]:
fig=base3()
plane3(fig,(1,-2,1),BLUE,0.24)
vec3(fig,(3,4,5),'<i>v</i>⃗<sub>1</sub>',BLUE)
vec3(fig,(2,0,-2),'<i>v</i>⃗<sub>2</sub>',ORANGE,offset=(-0.8,0.2,0.4))
fig.show()


---

## Normal vectors and plane equations

In [Chapter 2.1](02-01.ipynb), the coefficients of a line equation gave us a normal vector. In $\mathbb R^3$, a single linear equation describes a **plane**.

Let's keep working with the same plane $P$. So far, we've described it by telling you how to build vectors in it. We'd now like an equation that lets us check whether a given point is on the plane.

Remember that every vector in $P$ looks like

$$\begin{bmatrix}x\\y\\z\end{bmatrix}
=\begin{bmatrix}3a+2b\\4a\\5a-2b\end{bmatrix}.$$

If we substitute these entries into $x-2y+z$, we get

$$\begin{aligned}
x-2y+z
&=(3a+2b)-2(4a)+(5a-2b)\\
&=(3-8+5)a+(2-2)b\\
&=0.
\end{aligned}$$

No matter which $a$ and $b$ we pick, the result is $0$. So, **every vector in $P$ satisfies $x-2y+z=0$**. We still need to check the other direction: could this equation include any extra points that aren't in $P$?

::::{tip} Activity 2
Graph the equation

$$x-2y+z=0.$$

Is it the same as the plane $P$?

:::{tip} Solution
:class: dropdown

**Yes, the graph is exactly $P$.** Rearranging the equation gives $z=2y-x$, which we can graph as a plane:

```{figure} #plot-24-activity-2
:label: fig-24-activity-2
:class: course-caption
:alt: The plane z equals 2y minus x contains the two spanning vectors v1 and v2.

The graph of $z=2y-x$ is the plane $P$.
```

We already checked that every vector in $P$ satisfies the equation. For the other direction, take any point $(x,y,2y-x)$ on the graph. To write its position vector as $a\vec v_1+b\vec v_2$, the first two entries require

$$3a+2b=x,\qquad 4a=y.$$

The second equation gives $a=y/4$. Substituting this into the first gives

$$3\left(\frac y4\right)+2b=x
\quad\Longrightarrow\quad b=\frac x2-\frac{3y}{8}.$$

We chose $a$ and $b$ to match the first two entries. Let's check that the third entry works too:

$$5a-2b=\frac{5y}{4}-2\left(\frac x2-\frac{3y}{8}\right)
=\frac{5y}{4}-x+\frac{3y}{4}=2y-x,$$

exactly as needed. Thus,

$$\begin{bmatrix}x\\y\\2y-x\end{bmatrix}
=\frac y4\begin{bmatrix}3\\4\\5\end{bmatrix}
+\left(\frac x2-\frac{3y}{8}\right)\begin{bmatrix}2\\0\\-2\end{bmatrix}\in P.$$

This works for every point on the graph, so there are no extra points. The equation and the span really do describe the same plane.
:::
::::

Now, remember what we did with line equations in Chapter 2.1: we read off the coefficients to get a normal vector. Let's try the same thing here. The coefficients of $x-2y+z=0$ give us

$$\vec w=\begin{bmatrix}1\\-2\\1\end{bmatrix}.$$


In [6]:
fig=base3()
plane3(fig,(1,-2,1))
n=np.array([1,-2,1])
line3(fig,-2*n,2*n,PINK,width=3)
vec3(fig,(3,4,5),'<i>v</i>⃗<sub>1</sub>',BLUE)
vec3(fig,(2,0,-2),'<i>v</i>⃗<sub>2</sub>',ORANGE,offset=(-0.6,0.2,0.4))
vec3(fig,n,'<i>w</i>⃗',PINK,offset=(0.5,-0.5,0.4))
fig.update_layout(scene_camera=dict(eye=dict(x=2,y=-1.1,z=1.1)))
fig.show()


```{figure} #plot-24-normal
:label: fig-24-normal
:class: course-caption
:alt: The pink normal vector w is perpendicular to the blue plane P.

The vector
$\vec w=\begin{bmatrix}1\\-2\\1\end{bmatrix}$ is perpendicular to the plane
$P$.
```


To show that $\vec w$ is perpendicular to $P$, we need to show that it's orthogonal to **every vector in $P$**. There are infinitely many such vectors, but we only need two dot products to get started:

$$\vec v_1\cdot\vec w=3(1)+4(-2)+5(1)=0,$$

$$\vec v_2\cdot\vec w=2(1)+0(-2)+(-2)(1)=0.$$

Remember that every vector in $P$ is a linear combination of $\vec v_1$ and $\vec v_2$. Since both dot products above are $0$, the dot product of $\vec w$ with any of their linear combinations is also $0$:

$$\begin{aligned}
(a\vec v_1+b\vec v_2)\cdot\vec w
&=a(\vec v_1\cdot\vec w)+b(\vec v_2\cdot\vec w)\\
&=a(0)+b(0)=0.
\end{aligned}$$

That's why checking the two spanning vectors is enough. It tells us that $\vec w$ is perpendicular to the entire plane.

:::{note} Definition: Normal vector to a plane
A **normal vector** to a plane is a nonzero vector perpendicular to every direction in the plane.

For a plane with equation $ax+by+cz=d$, the coefficient vector $\begin{bmatrix}a\\b\\c\end{bmatrix}$ is a normal vector.
:::

For our plane through the origin, we can write

$$P=\left\{\begin{bmatrix}x\\y\\z\end{bmatrix}:
\vec w\cdot\begin{bmatrix}x\\y\\z\end{bmatrix}=0\right\}.$$

We'll write $P^\perp$ for the set of vectors orthogonal to every vector in $P$. Here it is the line

$$P^\perp=\operatorname{span}(\vec w).$$

This extends the perpendicular notation from Chapters 2.1–2.3. In $\mathbb R^2$, the vectors perpendicular to a line form another line. In $\mathbb R^3$, the vectors perpendicular to a plane form a line, while the vectors perpendicular to a line form a plane.

A normal vector is not unique. Using $3\vec w=\begin{bmatrix}3\\-6\\3\end{bmatrix}$ gives $3x-6y+3z=0$, which is the original equation multiplied by $3$. The plane does not change.

::::{tip} Activity 3
Which method of describing $P$ is easier:

- as the span of two vectors, or
- as the solution set of a single linear equation?

:::{tip} Solution
:class: dropdown

**It depends on what we want to do.** If we want to find vectors in $P$, the span description is convenient. If we want to check whether a particular vector is in $P$, the equation is convenient.

For example, choosing $a=2$ and $b=3$ immediately gives the vector $\begin{bmatrix}12\\8\\4\end{bmatrix}$ in $P$. If we are instead given that vector and asked whether it belongs to $P$, we can substitute into the equation:

$$12-2(8)+4=0.$$

The span tells us how to **build** vectors in the plane; the equation tells us how to **check** them. We'll keep using both perspectives.
:::
::::


---

## Lines as intersections of planes

Let's return to our original line $\ell=\operatorname{span}(\vec v_1)$. We'd like to describe it using equations, just as we did for $P$.

The equation $x-2y+z=0$ is satisfied by every vector on $\ell$, but it also includes all the other vectors in $P$. It gives us too many points! We'll use a second equation to keep only the points on the line.

One linear equation in $\mathbb R^3$ with a nonzero normal vector defines a plane. To define a line, we need two equations whose normal vectors are not scalar multiples of each other.

For example, consider the line

$$
\ell
=
\operatorname{span}
\left(
\begin{bmatrix}
3\\
4\\
5
\end{bmatrix}
\right).
$$

One equation satisfied by every vector on this line is

$$
x-2y+z=0.
$$

We need another equation that is independent of the first.

There are many ways to do this. For example, each of the following equations
is satisfied by every vector on $\ell$:

$$
4x-3y=0,
$$

$$
5y-4z=0,
$$

$$
5x-3z=0,
$$

or

$$
-5x+5y-z=0.
$$

Let

$$
\vec w=
\begin{bmatrix}
1\\
-2\\
1
\end{bmatrix}
$$

and choose

$$
\vec w'=
\begin{bmatrix}
-5\\
5\\
-1
\end{bmatrix}.
$$

The plane perpendicular to $\vec w$ is

$$
P=\left\{\vec x\in\mathbb{R}^3:\vec x\cdot\vec w=0\right\},
$$

and the plane perpendicular to $\vec w'$ is

$$
P'=\left\{\vec x\in\mathbb{R}^3:\vec x\cdot\vec w'=0\right\}.
$$

Requiring **both** equations to hold means keeping only the points shared by $P$ and $P'$, their intersection $P\cap P'$. The figure suggests that this is our line $\ell$; next, we'll check it algebraically.


In [7]:
fig=base3()
plane3(fig,(1,-2,1),BLUE,0.28)
plane3(fig,(-5,5,-1),ORANGE,0.28)
v=np.array([3,4,5])
line3(fig,-1.1*v,1.3*v,PINK,width=8)
vec3(fig,v,'<i>v</i>⃗<sub>1</sub>',PINK,offset=(0.6,0.3,0.5))
vec3(fig,(1,-2,1),'<i>w</i>⃗',BLUE,offset=(0.2,-0.3,0.4))
vec3(fig,(-5,5,-1),"<i>w</i>⃗′",ORANGE,offset=(0.2,0.2,0.5))
fig.show()


```{figure} #plot-24-intersection
:label: fig-24-intersection
:class: course-caption
:alt: The blue and orange planes intersect along the pink line ell.

The line
$\ell=\operatorname{span}\left(\begin{bmatrix}3\\4\\5\end{bmatrix}\right)$
is the intersection of two planes through the origin.
```


Let's find all points that satisfy both equations. The first equation gives $z=2y-x$, so we can substitute that expression for $z$ into the second:

$$\begin{aligned}
-5x+5y-(2y-x)&=0,\\
-4x+3y&=0,\\
y&=\frac43x.
\end{aligned}$$

Once we know $x$, both other coordinates are determined: $y=\frac43x$ and $z=2(\frac43x)-x=\frac53x$. To make this look like our original parametric description, write $x=3t$. Then

$$\begin{bmatrix}x\\y\\z\end{bmatrix}
=\begin{bmatrix}3t\\4t\\5t\end{bmatrix}
=t\begin{bmatrix}3\\4\\5\end{bmatrix}.$$

So the simultaneous equations

$$\begin{cases}x-2y+z=0,\\-5x+5y-z=0\end{cases}$$

describe exactly the line $\ell$. In set notation,

$$\ell=P\cap P'
=\{\vec x\in\mathbb R^3:\vec x\cdot\vec w=0\text{ and }\vec x\cdot\vec w'=0\},$$

where $\vec x=\begin{bmatrix}x\\y\\z\end{bmatrix}$.

The two normal vectors must be independent. Multiplying the first equation by $3$, for example, would give the same plane again, so it would not narrow the intersection to a line. Any two independent normal vectors in $\ell^\perp$ give a pair of homogeneous equations describing $\ell$.


---

## Affine lines and planes

Everything we've drawn so far passes through the origin. To move a line or plane somewhere else, we can use the same idea as in [Chapter 2.1](02-01.ipynb): **add a fixed vector $\vec p$ to every vector in the set**.

:::{note} Affine lines and planes
An **affine line** has the form

$$\vec p+\operatorname{span}(\vec v)
=\{\vec p+t\vec v:t\in\mathbb R\},\qquad\vec v\ne\vec0.$$

An **affine plane** has the form

$$\vec p+\operatorname{span}(\vec v_1,\vec v_2)
=\{\vec p+a\vec v_1+b\vec v_2:a,b\in\mathbb R\},$$

where $\vec v_1$ and $\vec v_2$ are independent.

The fixed vector $\vec p$ gives a starting point. The spanning vectors give directions along the line or plane.
:::

For example, take $\vec p=\begin{bmatrix}0\\0\\3\end{bmatrix}$ and translate our plane $P$ by $\vec p$. Its parametric description becomes

$$\begin{bmatrix}x\\y\\z\end{bmatrix}
=\begin{bmatrix}0\\0\\3\end{bmatrix}
+a\begin{bmatrix}3\\4\\5\end{bmatrix}
+b\begin{bmatrix}2\\0\\-2\end{bmatrix},\qquad a,b\in\mathbb R.$$

Translation preserves the directions in the plane, so $\vec w=\begin{bmatrix}1\\-2\\1\end{bmatrix}$ is still a normal vector. A point with position vector $\vec x$ is on the translated plane exactly when $\vec x-\vec p$ lies in $P$. Therefore,

$$\begin{aligned}
\vec w\cdot(\vec x-\vec p)&=0,\\
\vec w\cdot\vec x&=\vec w\cdot\vec p,\\
x-2y+z&=3.
\end{aligned}$$

```{figure} #plot-24-affine
:label: fig-24-affine
:class: course-caption
:alt: The blue plane x minus 2y plus z equals zero is translated upward by p to the orange plane x minus 2y plus z equals three.

Adding $\vec p=\begin{bmatrix}0\\0\\3\end{bmatrix}$ translates $P$ to the parallel plane $x-2y+z=3$.
```

More generally, a plane has equation $ax+by+cz=d$, where the normal vector $\begin{bmatrix}a\\b\\c\end{bmatrix}$ is nonzero. If $d=0$, the equation is **homogeneous** and the plane passes through the origin. If $d\ne0$, it is **nonhomogeneous** and the plane does not pass through the origin.

The same idea works for a line. Adding $\vec p$ to each vector on $\ell=\operatorname{span}(\vec v_1)$ gives

$$\begin{bmatrix}x\\y\\z\end{bmatrix}
=\begin{bmatrix}0\\0\\3\end{bmatrix}
+t\begin{bmatrix}3\\4\\5\end{bmatrix},\qquad t\in\mathbb R.$$

Again, we haven't changed the direction of the line, so we can keep the same normal vectors $\vec w$ and $\vec w'$. Taking their dot products with $\vec p$ gives the new right-hand sides, $3$ and $-3$:

$$\begin{cases}x-2y+z=3,\\-5x+5y-z=-3.\end{cases}$$

An affine line in $\mathbb R^3$ is the intersection of two planes with independent normal vectors. Translating changes the right-hand sides of their equations while preserving the direction of the line.


In [8]:
fig=base3()
plane3(fig,(1,-2,1),BLUE,0.18)
plane3(fig,(1,-2,1),ORANGE,0.18,d=3)
vec3(fig,(0,0,3),'<i>p</i>⃗',PINK,offset=(0.4,-0.5,0.3))
fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers',
                         marker=dict(color='black',size=4),hoverinfo='skip'))
fig.show()
